# Notebook 01 — Baseline DQN on CartPole-v1

APBIO project, 2025/26.

Before wiring up the Hopfield replay buffer, I need a plain DQN that actually
learns on CartPole. This notebook does the minimum: set up Drive, install
packages, train a single-seed DQN, save the log, and plot the learning curve.

If this works, I can move on to the Hopfield module in notebook 02.

Runtime: roughly 5–8 minutes on a CPU runtime.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/apbio_project')
(PROJECT_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

## 2. Install packages

Colab already has PyTorch. I just add Gymnasium and Stable-Baselines3.

In [ ]:
%pip install -q 'gymnasium>=0.29' 'stable-baselines3>=2.3'

In [ ]:
import gymnasium as gym
import stable_baselines3 as sb3
import torch, numpy as np

print('gymnasium:', gym.__version__)
print('sb3      :', sb3.__version__)
print('torch    :', torch.__version__)

## 3. Seed helper

The project spec asks for results averaged over at least 5 seeds. This helper
sets every RNG I can think of so the runs are reproducible.

In [ ]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(0)

## 4. Check the environment

CartPole-v1: 4-dim state, 2 discrete actions, max return 500.

In [ ]:
env = gym.make('CartPole-v1')
obs, _ = env.reset(seed=0)
print('obs space   :', env.observation_space)
print('action space:', env.action_space)
print('sample obs  :', obs)
env.close()

## 5. Per-episode CSV logger

The project requires raw per-seed CSV logs. SB3 keeps finished episodes in
`model.ep_info_buffer`; I just drain that buffer periodically and write new
rows to a CSV.

In [ ]:
import csv
from pathlib import Path
from stable_baselines3.common.callbacks import BaseCallback

class EpisodeLogger(BaseCallback):
    def __init__(self, csv_path):
        super().__init__()
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self._ep = 0
        self._seen = 0
        with self.csv_path.open('w', newline='') as f:
            csv.writer(f).writerow(['episode', 'timestep', 'return', 'length'])

    def _on_step(self):
        buf = list(self.model.ep_info_buffer)
        new = buf[self._seen:]
        if new:
            rows = []
            for ep in new:
                self._ep += 1
                rows.append([self._ep, self.num_timesteps, float(ep['r']), int(ep['l'])])
            with self.csv_path.open('a', newline='') as f:
                csv.writer(f).writerows(rows)
            self._seen = len(buf)
        return True

## 6. Train — single seed smoke test

Hyperparameters below come from the official SB3 RL Zoo configuration for
`dqn` on `CartPole-v1` (see https://huggingface.co/sb3/dqn-CartPole-v1). With
the tuned settings, DQN reaches ~500 return within 50k steps. I use those as
defaults.

My earlier attempt with `lr=5e-4, gradient_steps=1, 30k steps` did not learn
(final 10-ep mean was ~15), so I switched to the zoo defaults.

In [ ]:
import time
from stable_baselines3 import DQN

SEED = 0
TOTAL_TIMESTEPS = 50_000
LOG_PATH = PROJECT_ROOT / 'logs' / f'baseline_seed{SEED}.csv'

set_seed(SEED)
env = gym.make('CartPole-v1')

# RL Zoo tuned hyperparameters for DQN on CartPole-v1.
model = DQN(
    policy='MlpPolicy',
    env=env,
    learning_rate=2.3e-3,
    buffer_size=100_000,
    learning_starts=1_000,
    batch_size=64,
    tau=1.0,
    gamma=0.99,
    train_freq=256,
    gradient_steps=128,
    target_update_interval=10,
    exploration_fraction=0.16,
    exploration_final_eps=0.04,
    policy_kwargs=dict(net_arch=[256, 256]),
    seed=SEED,
    verbose=0,
)

callback = EpisodeLogger(csv_path=str(LOG_PATH))

t0 = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback, progress_bar=True)
print(f'done in {time.time() - t0:.1f} s')
print('log:', LOG_PATH)

## 7. Plot the learning curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(LOG_PATH)
df['smooth'] = df['return'].rolling(20, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df['episode'], df['return'], alpha=0.25, label='episode return')
ax.plot(df['episode'], df['smooth'], linewidth=2, label='20-ep rolling mean')
ax.axhline(500, color='grey', linestyle='--', linewidth=1, label='max (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Return')
ax.set_title(f'Baseline DQN on CartPole-v1 (seed {SEED})')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()

out = PROJECT_ROOT / 'figures' / f'baseline_seed{SEED}.png'
fig.savefig(out, dpi=150)
plt.show()
print('figure:', out)

## 8. Quick sanity check

I want to see two things:

1. The agent clearly learns — the last 10-episode mean should be much higher
   than the first 10.
2. The agent gets close to the 500 ceiling at the end.

In [ ]:
print('episodes            :', len(df))
print('first 10 ep mean    :', round(df['return'].head(10).mean(), 1))
print('last 10 ep mean     :', round(df['return'].tail(10).mean(), 1))
print('max return          :', int(df['return'].max()))

## Next

If the last-10 mean is comfortably above 400, the stack works. Notebook 02
will implement the Hopfield Network and test it on synthetic patterns before
plugging it into the replay buffer.